Quiz 2 write a code which will detect the value of dollar bill. dataset is in zip file you have to first create a test folder in test data you will crease the same folders as in zip file  and then cut few images from each folder in. zip and place it its corresponding folder in test folder.After that you will train some CNN model or use any Other method and test its accuracy on test data.make sure the images which you place in Test folder should not be present in train folder

In [ ]:
#Step 1
import zipfile, os, shutil

zip_path = "/content/Bill_dataset.zip"
extract_path = "Bill_data_raw"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

# Remove Mac junk folder
if os.path.exists("__MACOSX"):
    shutil.rmtree("__MACOSX")

print("Dataset extracted")


Dataset extracted


Keep ONLY VALID IMAGE FILES (.tif)

In [ ]:
VALID_EXT = ('.tif', '.tiff')

def clean_folder(path):
    for root, dirs, files in os.walk(path):
        for f in files:
            if not f.lower().endswith(VALID_EXT):
                os.remove(os.path.join(root, f))

clean_folder("Bill_data_raw")
print("Cleaned non-image files")


Cleaned non-image files


Define TRAIN & TEST FOLDERS (CUT IMAGES)

In [ ]:
import random

source_dir = "Bill_data_raw/Bill_dataset"
train_dir = "train_data"
test_dir = "test_data"

os.makedirs(train_dir, exist_ok=True)
os.makedirs(test_dir, exist_ok=True)

TEST_IMAGES_PER_CLASS = 5


CREATE TEST FOLDER & CUT IMAGES

In [ ]:
for cls in os.listdir(source_dir):
    cls_path = os.path.join(source_dir, cls)
    if not os.path.isdir(cls_path):
        continue

    os.makedirs(os.path.join(train_dir, cls), exist_ok=True)
    os.makedirs(os.path.join(test_dir, cls), exist_ok=True)

    images = os.listdir(cls_path)
    random.shuffle(images)

    test_imgs = images[:TEST_IMAGES_PER_CLASS]
    train_imgs = images[TEST_IMAGES_PER_CLASS:]

    for img in test_imgs:
        shutil.move(
            os.path.join(cls_path, img),
            os.path.join(test_dir, cls, img)
        )

    for img in train_imgs:
        shutil.move(
            os.path.join(cls_path, img),
            os.path.join(train_dir, cls, img)
        )


In [ ]:
from PIL import Image
import os

def convert_tif_to_jpg(input_dir):
    converted = 0
    for root, dirs, files in os.walk(input_dir):
        for file in files:
            if file.lower().endswith(('.tif', '.tiff')):
                tif_path = os.path.join(root, file)
                jpg_path = tif_path.replace('.tif', '.jpg').replace('.tiff', '.jpg')

                try:
                    img = Image.open(tif_path).convert("RGB")
                    img.save(jpg_path, "JPEG")
                    os.remove(tif_path)  # remove original tif
                    converted += 1
                except:
                    print("Failed:", tif_path)

    print(f"Converted {converted} TIFF images to JPG")


In [ ]:
convert_tif_to_jpg(train_dir)
convert_tif_to_jpg(test_dir)


Converted 141 TIFF images to JPG
Converted 20 TIFF images to JPG


1)Images are CUT, not copied
2) No test image remains in training
 This line satisfies the quiz condition fully

In [ ]:
import tensorflow as tf

IMG_SIZE = (224, 224)
BATCH_SIZE = 8

train_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    test_dir,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)


Found 141 files belonging to 4 classes.
Found 20 files belonging to 4 classes.


Normalize the data

In [ ]:
train_ds = train_ds.map(lambda x, y: (x/255.0, y))
test_ds = test_ds.map(lambda x, y: (x/255.0, y))


Simple CNN model

In [ ]:
from tensorflow.keras import layers, models
import os

# Get the number of classes by counting subdirectories in train_dir
num_classes = len([name for name in os.listdir(train_dir) if os.path.isdir(os.path.join(train_dir, name))])

model = models.Sequential([
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(224,224,3)),
    layers.MaxPooling2D(),

    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D(),

    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(num_classes, activation='softmax')
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


TRAIN (THIS WILL WORK)

In [ ]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)


In [ ]:
model.fit(
    train_ds,
    epochs=5,
    validation_data=test_ds
)


Epoch 1/5
18/18 ━━━━━━━━━━━━━━━━━━━━ 20s 979ms/step - accuracy: 0.5078 - loss: 7.1473 - val_accuracy: 0.8500 - val_loss: 0.9845
Epoch 2/5
18/18 ━━━━━━━━━━━━━━━━━━━━ 19s 948ms/step - accuracy: 0.9797 - loss: 0.0920 - val_accuracy: 0.9000 - val_loss: 0.1836
Epoch 3/5
18/18 ━━━━━━━━━━━━━━━━━━━━ 20s 936ms/step - accuracy: 1.0000 - loss: 0.0131 - val_accuracy: 0.9000 - val_loss: 0.1574
Epoch 4/5
18/18 ━━━━━━━━━━━━━━━━━━━━ 17s 920ms/step - accuracy: 1.0000 - loss: 5.8285e-04 - val_accuracy: 0.9500 - val_loss: 0.1265
Epoch 5/5
18/18 ━━━━━━━━━━━━━━━━━━━━ 20s 909ms/step - accuracy: 1.0000 - loss: 8.3764e-05 - val_accuracy: 0.9500 - val_loss: 0.1280


TEST ACCURACY

In [ ]:
loss, acc = model.evaluate(test_ds)
print("Test Accuracy:", acc)


3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 115ms/step - accuracy: 0.9281 - loss: 0.1773
Test Accuracy: 0.949999988079071
